In [1]:
import pandas as pd
import sqlite3 as sq

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.options.display.float_format = '{:,.2f}'.format

In [3]:
with sq.connect('db.sqlite3') as con:
    db_data = pd.read_sql_query('select * from frc_user;', con)

In [5]:
db_data

,id,frc,user,email,login
0,1,admin,Кондратенко Денис Анатольевич,d.kondratenko@rt-techpriemka.ru,d.kondratenko
1,2,admin,Коньшин Дмитрий Сергеевич,d.konshin@rt-techpriemka.ru,d.konshin
2,3,admin,Чайковский Илья Сергеевич,i.chaykovskiy@rt-techpriemka.ru,i.chaykovskiy
3,4,Административно-хозяйственное обеспечение,Наумова Ирина Сергеевна,i.naumova@rt-techpriemka.ru,i.naumova
4,5,Административно-хозяйственное обеспечение,Шмелев Алексей Викторович,a.shmelev@rt-techpriemka.ru,a.shmelev
5,6,Бухгалтерия,Кузнецова Елена Анатольевна,ea.kuznetsova@rt-techpriemka.ru,ea.kuznetsova
6,7,Бухгалтерия,Фролов Евгений Валерьевич,e.frolov@rt-techpriemka.ru,e.frolov
7,8,Внутренняя система менеджмента качества,Михайлов Павел Михайлович,p.mihaylov@rt-techpriemka.ru,p.mihaylov
8,9,Инновации и инжиниринг,Дряев Георгий Георгиевич,g.dryaev@rt-techpriemka.ru,g.dryaev
9,10,Инновации и инжиниринг,Ульянов Евгений Ильич,e.ulyanov@rt-techpriemka.ru,e.ulyanov


In [6]:
exl_data = pd.read_excel('new_frc_users.xlsx', sheet_name='для ввода прогноза')

In [7]:
exl_data

,Unnamed: 0,ЦФО,Ф,И,О,почта,Ф.1,И.1,О.1,почта.1
0,4,Инновации и инжиниринг,Ульянов,Евгений,Ильич,e.ulyanov@rt-techpriemka.ru,Дряев,Георгий,Георгиевич,g.dryaev@rt-techpriemka.ru
1,10,Направление продаж,Моисеенко,Виталий,Евгеньевич,v.moiseenko@rt-techpriemka.ru,Запорожский,Вячеслав,Александрович,v.zaporozhskiy@rt-techpriemka.ru
2,14,Оценка и технический контроль,Ткачев,Сергей,Владимирович,s.tkachev@rt-techpriemka.ru,NaN,NaN,NaN,NaN
3,16,Стратегия и инвестиции,Виноградов,Всеволод,Владимирович,v.vinogradov@rt-techpriemka.ru,Малиновский,Ярослав,Владимирович,ya.malinovskiy@rt-techpriemka.ru
4,18,Управление проектами и цифровизацией,Леонов,Александр,Владимирович,a.leonov@rt-techpriemka.ru,NaN,NaN,NaN,NaN
5,19,Центр качества поставок,Захарова,Алена,Станиславовна,a.zaharova@rt-techpriemka.ru,NaN,NaN,NaN,NaN
6,20,Центр компетенции и системы управления качеством ГК Ростех,Никулин,Василий,Семенович,v.nikulin@rt-techpriemka.ru,NaN,NaN,NaN,NaN
7,21,Центр обучения,Ефремов,Василий,Петрович,v.efremov@rtqualityplus.ru,Кошелева,Елизавета,Робертовна,e.kosheleva@rt-techpriemka.ru
8,22,Центр сертификации,Ефремов,Василий,Петрович,v.efremov@rtqualityplus.ru,Грудцов,Иван,Александрович,i.grudcov@rt-techpriemka.ru


In [8]:
user_to_insert = pd.concat([
( 
    exl_data
    .assign(user = lambda x: x['Ф'] + ' ' + x['И'] + ' ' + x['О'])
    .rename({'ЦФО': 'frc', 'почта': 'email'}, axis=1)
    .dropna(subset='email')
    .assign(login = lambda x: x['email'].apply(lambda x: x.split('@')[0]))
    [['frc', 'user', 'email', 'login']]
),
( 
    exl_data
    .assign(user = lambda x: x['Ф.1'] + ' ' + x['И.1'] + ' ' + x['О.1'])
    .rename({'ЦФО': 'frc', 'почта.1': 'email'}, axis=1)
    .dropna(subset='email')
    .assign(login = lambda x: x['email'].apply(lambda x: x.split('@')[0]))
    [['frc', 'user', 'email', 'login']]
)
]).iloc[1:,:]

In [10]:
user_to_insert.to_sql('frc_user', con, if_exists='append', index=False)

13